# Predator-prey dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gstonge/csi4124-sys5110/blob/main/notebooks/predator-prey.ipynb)
[![Open in JupyterLite](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://www.gstonge.ca/csi4124-sys5110/lite/notebooks/index.html?path=predator-prey.ipynb)

This notebook integrates two-dimensional predator-prey models and visualizes the state evolution: the population densities over time and the trajectories in the phase portrait. Widgets let you switch between models and change the model parameters, the initial conditions and the integration time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, fixed, ToggleButtons

## Model definition

Below we parametrize two versions of predator-prey dynamics: 1) the classic Lotka–Volterra model, and 2) the Rosenzweig–MacArthur model, which extends Lotka–Volterra with a carrying capacity for the prey and a saturation of predation when prey become plentiful. We provide vector fields for both.

### Classic Lotka–Volterra

Here we consider densities of prey $x(t)$ and predators $y(t)$ per unit of an appropriately chosen area, such that $x, y \sim 1$.
These quantities obey the following system of ODEs:

$$
\frac{dx}{dt} = \alpha x - \beta x y, \qquad
\frac{dy}{dt} = \delta x y - \gamma y
$$

In [ ]:
def vector_field_lv(t, z, alpha=1.0, beta=1.0, delta=1.0, gamma=1.0):
    prey, predator = z
    dprey = alpha * prey - beta * prey * predator
    dpredator = delta * prey * predator - gamma * predator
    return [dprey, dpredator]



parameter_ranges_lv = {
    'alpha':(0.0, 2.0, 0.1),
    'beta':(0.0, 2.0, 0.1),
    'delta':(0.0, 2.0, 0.1),
    'gamma':(0.0, 2.0, 0.1)
}

### Rosenzweig–MacArthur

Adding a carrying capacity for prey and a saturation effect for predation when prey are abundant, we have the following system of ODEs:

$$
\frac{dx}{dt} = \alpha x \left(1 - \frac{x}{K}\right) - \frac{\beta x y}{1 + x/\eta}, \qquad
\frac{dy}{dt} = \frac{\delta x y}{1 + x/\eta} - \gamma y
$$

In [ ]:
def vector_field_rm(t, z, alpha=1.0, beta=1.0, delta=1.0, gamma=1.0, eta=5, K=10):
    prey, predator = z
    dprey = alpha * prey * (1 - prey/K) - beta * prey * predator/(1 + prey / eta)
    dpredator = delta * prey * predator / (1 + prey / eta) - gamma * predator
    return [dprey, dpredator]

parameter_ranges_rm = {
    'alpha':(0.0, 2.0, 0.1),
    'beta':(0.0, 2.0, 0.1),
    'delta':(0.0, 2.0, 0.1),
    'gamma':(0.0, 2.0, 0.1),
    'eta':(0.1, 30, 0.1), #eating saturation for predator
    'K':(0.1,30,0.1) #carrying capacity for prey
}

In [ ]:
vector_fields = {
    "Lotka-Volterra": vector_field_lv,
    "Rosenzweig-MacArthur": vector_field_rm,
}

parameter_ranges = {
    "Lotka-Volterra": parameter_ranges_lv,
    "Rosenzweig-MacArthur": parameter_ranges_rm,
}

## Trajectory visualization

Below we focus on visualizing the trajectories of the state variables.

In [ ]:
def integrate(f, z0, t_max, **params):
    t = np.linspace(0, t_max, 1000)
    solution = solve_ivp(lambda t, z: f(t, z, **params), (0, t_max), z0, t_eval=t, rtol=1e-8)
    return solution.t, solution.y[0], solution.y[1]


def plot_trajectories(f, initial_conditions, t_max, **params):
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 5))

    for pair in initial_conditions.split(";"):
        z0 = [float(value) for value in pair.split(",")]
        t, x, y = integrate(f, z0, t_max, **params)
        ax1.plot(t, x, color="tab:blue")
        ax2.plot(t, y, color="tab:red")

    ax1.set_ylabel("prey")
    ax2.set_ylabel("predator")
    ax2.set_xlabel("time")
    plt.show()

In [ ]:
def show_trajectories(model):
    interact(
        plot_trajectories,
        f=fixed(vector_fields[model]),
        initial_conditions="1, 0.5",
        t_max=(1, 100, 1),
        **parameter_ranges[model],
    )

interact(show_trajectories, model=ToggleButtons(options=vector_fields.keys()));

## Phase portrait

Here we append a visualization of the phase portrait alongside the trajectories of the state variables.

### Using streamplot

In [ ]:
def plot_phase_portrait_streamplot(f, initial_conditions, t_max, **params):
    """Here we use streamplot, with curved arrows representing integrated local trajectories in the state space"""
    fig = plt.figure(figsize=(13, 5))
    ax1 = fig.add_subplot(2, 2, 1)
    ax2 = fig.add_subplot(2, 2, 3, sharex=ax1)
    ax3 = fig.add_subplot(1, 2, 2)

    all_x = []
    all_y = []
    for i, pair in enumerate(initial_conditions.split(";")):
        z0 = [float(value) for value in pair.split(",")]
        t, x, y = integrate(f, z0, t_max, **params)
        ax1.plot(t, x, color="tab:blue")
        ax2.plot(t, y, color="tab:red")
        ax3.plot(x, y, color="black")
        ax3.plot(x[0], y[0], "o", color=f"C{i+1}", markersize=8, zorder=3, label=pair.strip())
        ax3.plot(x[-1], y[-1], "*", color=f"C{i+1}", markersize=14, zorder=3)
        all_x.append(x)
        all_y.append(y)

    x_grid = np.linspace(0, 1.2 * np.max(all_x), 30)
    y_grid = np.linspace(0, 1.2 * np.max(all_y), 30)
    X, Y = np.meshgrid(x_grid, y_grid)
    U, V = f(0, [X, Y], **params)
    ax3.streamplot(X, Y, U, V, color="lightgray")

    ax3.plot([], [], "o", color="gray", markersize=8, label="start")
    ax3.plot([], [], "*", color="gray", markersize=12, label="end")
    ax3.legend(loc="lower center", bbox_to_anchor=(0.5, 1), ncol=6)

    ax1.set_ylabel("prey")
    ax2.set_ylabel("predator")
    ax2.set_xlabel("time")
    ax3.set_xlabel("prey")
    ax3.set_ylabel("predator")
    plt.show()

In [ ]:
def show_phase_portrait_streamplot(model):
    interact(
        plot_phase_portrait_streamplot,
        f=fixed(vector_fields[model]),
        initial_conditions="1, 0.5",
        t_max=(1, 100, 1),
        **parameter_ranges[model],
    )

interact(show_phase_portrait_streamplot, model=ToggleButtons(options=vector_fields.keys()));

### Using quiver

In [ ]:
def plot_phase_portrait_quiver(f, initial_conditions, t_max, **params):
    """Here we use quiver, with straight arrows showing the direction and speed of the vector field on a grid"""
    fig = plt.figure(figsize=(13, 5))
    ax1 = fig.add_subplot(2, 2, 1)
    ax2 = fig.add_subplot(2, 2, 3, sharex=ax1)
    ax3 = fig.add_subplot(1, 2, 2)

    all_x = []
    all_y = []
    for i, pair in enumerate(initial_conditions.split(";")):
        z0 = [float(value) for value in pair.split(",")]
        t, x, y = integrate(f, z0, t_max, **params)
        ax1.plot(t, x, color="tab:blue")
        ax2.plot(t, y, color="tab:red")
        ax3.plot(x, y, color="black")
        ax3.plot(x[0], y[0], "o", color=f"C{i+1}", markersize=8, zorder=3, label=pair.strip())
        ax3.plot(x[-1], y[-1], "*", color=f"C{i+1}", markersize=14, zorder=3)
        all_x.append(x)
        all_y.append(y)

    x_grid = np.linspace(0, 1.2 * np.max(all_x), 20)
    y_grid = np.linspace(0, 1.2 * np.max(all_y), 20)
    X, Y = np.meshgrid(x_grid, y_grid)
    U, V = f(0, [X, Y], **params)
    ax3.quiver(X, Y, U, V, color="gray")

    ax3.plot([], [], "o", color="gray", markersize=8, label="start")
    ax3.plot([], [], "*", color="gray", markersize=12, label="end")
    ax3.legend(loc="lower center", bbox_to_anchor=(0.5, 1), ncol=6)

    ax1.set_ylabel("prey")
    ax2.set_ylabel("predator")
    ax2.set_xlabel("time")
    ax3.set_xlabel("prey")
    ax3.set_ylabel("predator")
    plt.show()

In [ ]:
def show_phase_portrait_quiver(model):
    interact(
        plot_phase_portrait_quiver,
        f=fixed(vector_fields[model]),
        initial_conditions="1, 0.5",
        t_max=(1, 100, 1),
        **parameter_ranges[model],
    )

interact(show_phase_portrait_quiver, model=ToggleButtons(options=vector_fields.keys()));